In [1]:
import sys
print(sys.executable)

import pandas as pd
import numpy as np
import sklearn 
import lightgbm
import jupyter
print("Pandas imported successfully!")



/home/ahmed/data-science-bootcamp/favorita-grocery-sales-forecasting/.venv/bin/python
Pandas imported successfully!


In [2]:
# Read only the first 5 rows—takes 0.01 seconds and uses almost zero RAM
sample_df = pd.read_csv('data/train.csv', nrows=5)
print(sample_df)
print("\nDefault Data Types:\n", sample_df.dtypes)

   id        date  store_nbr  item_nbr  unit_sales  onpromotion
0   0  2013-01-01         25    103665         7.0          NaN
1   1  2013-01-01         25    105574         1.0          NaN
2   2  2013-01-01         25    105575         2.0          NaN
3   3  2013-01-01         25    108079         1.0          NaN
4   4  2013-01-01         25    108701         1.0          NaN

Default Data Types:
 id               int64
date               str
store_nbr        int64
item_nbr         int64
unit_sales     float64
onpromotion    float64
dtype: object


In [4]:
dtypes = {
    'id': 'int32',            # Row ID fits inside 32-bit integer (up to ~2.1 billion)
    'store_nbr': 'int8',      # Only 54 stores; fits inside 8-bit integer (-128 to 127)
    'item_nbr': 'int32',      # Item IDs go up to ~2.1 million; needs 32-bit integer
    'unit_sales': 'float32',  # Decimal sales quantities; 32-bit float saves 50% RAM over float64
    'onpromotion': 'boolean'  # Special Pandas nullable boolean that handles True, False, AND NaN!
}

In [11]:
dtypes = {
    'id': 'int32',            # 4 bytes per row instead of 8 bytes (int64)
    'store_nbr': 'int8',      # 1 byte per row instead of 8 bytes (int64)
    'item_nbr': 'int32',      # 4 bytes per row instead of 8 bytes (int64)
    'unit_sales': 'float32',  # 4 bytes per row instead of 8 bytes (float64)
    'onpromotion': 'boolean'  # 2 bytes per row (nullable boolean) instead of 8 bytes
}

# Load the full dataset (takes ~1 to 2 minutes for 125.5 million rows)
df_train = pd.read_csv(
    'data/train.csv',
    dtype=dtypes,
    parse_dates=['date'],
    usecols=['date', 'store_nbr', 'item_nbr', 'unit_sales', 'onpromotion']
)

# Check total rows, data types, and exact RAM usage
print(f"Total rows loaded: {len(df_train):,}")
print(df_train.info(memory_usage='deep'))

Total rows loaded: 125,497,040
<class 'pandas.DataFrame'>
RangeIndex: 125497040 entries, 0 to 125497039
Data columns (total 5 columns):
 #   Column       Dtype         
---  ------       -----         
 0   date         datetime64[us]
 1   store_nbr    int8          
 2   item_nbr     int32         
 3   unit_sales   float32       
 4   onpromotion  boolean       
dtypes: boolean(1), datetime64[us](1), float32(1), int32(1), int8(1)
memory usage: 2.2 GB
None


In [12]:
# 1. Check how many True, False, and NaN values exist
print("Before cleaning onpromotion:")
print(df_train['onpromotion'].value_counts(dropna=False))

Before cleaning onpromotion:
onpromotion
False    96028767
<NA>     21657651
True      7810622
Name: count, dtype: Int64


In [ ]:
# Fill NaN with False, then cast to 1-byte integer (0 = False, 1 = True)
df_train['onpromotion'] = df_train['onpromotion'].fillna(False).astype('int8')

print("\nAfter cleaning onpromotion:")
print(df_train['onpromotion'].value_counts())


After cleaning onpromotion:
onpromotion
0    117686418
1      7810622
Name: count, dtype: int64


In [14]:
# 1. Check the minimum value and count how many rows have negative sales
neg_count = (df_train['unit_sales'] < 0).sum()
print(f"Min unit_sales before cleaning: {df_train['unit_sales'].min()}")
print(f"Number of negative sales rows: {neg_count:,}")

# 2. Method A: Clip all negative values to 0.0
df_train['unit_sales'] = df_train['unit_sales'].clip(lower=0)

print(f"Min unit_sales after cleaning: {df_train['unit_sales'].min()}")

Min unit_sales before cleaning: -15372.0
Number of negative sales rows: 7,795
Min unit_sales after cleaning: 0.0


In [15]:
# 1. Verify no other columns have missing values
print("Missing values per column:\n", df_train.isna().sum())

# 2. Check the date range and positive max outlier
print(f"\nDate range: {df_train['date'].min()} to {df_train['date'].max()}")
print(f"Max unit_sales in a single day: {df_train['unit_sales'].max():,}")

# 3. Check promotion rate by year (Notice 2013 is 0% because it was all <NA>!)
print("\nPromotion rate by year:")
print(df_train.groupby(df_train['date'].dt.year)['onpromotion'].mean())

Missing values per column:
 date           0
store_nbr      0
item_nbr       0
unit_sales     0
onpromotion    0
dtype: int64

Date range: 2013-01-01 00:00:00 to 2017-08-15 00:00:00
Max unit_sales in a single day: 89,440.0

Promotion rate by year:
date
2013    0.000000
2014    0.020614
2015    0.039020
2016    0.099761
2017    0.115491
Name: onpromotion, dtype: float64


In [17]:
import gc  # Garbage Collector to free RAM immediately

# Keep 2016 and 2017 data (clean promotions, modern store catalog, < 1 GB RAM)
df_train = df_train[df_train['date'] >= '2016-01-01'].reset_index(drop=True)
gc.collect()  # Tells Python to immediately free the deleted 2013-2015 memory

print(f"Filtered rows (2016-2017): {len(df_train):,}")
print(f"New RAM usage: {df_train.memory_usage(deep=True).sum() / 1e9:.2f} GB")

# Save to Parquet so reloading takes 2 seconds tomorrow
df_train.to_parquet('data/train_clean_2016_2017.parquet', index=False)

Filtered rows (2016-2017): 59,038,132
New RAM usage: 1.06 GB


In [18]:
# 1. Define memory-efficient data types for test.csv
test_dtypes = {
    'id': 'int32',            # Needed in test.csv for the final Kaggle submission file!
    'store_nbr': 'int8',
    'item_nbr': 'int32',
    'onpromotion': 'boolean'
}

# 2. Load test.csv
df_test = pd.read_csv(
    'data/test.csv',
    dtype=test_dtypes,
    parse_dates=['date']
)

# 3. Clean onpromotion exactly like we did in df_train
df_test['onpromotion'] = df_test['onpromotion'].fillna(False).astype('int8')

print(f"Test rows: {len(df_test):,}")
print(f"Test date range: {df_test['date'].min()} to {df_test['date'].max()}")
display(df_test.head())

Test rows: 3,370,464
Test date range: 2017-08-16 00:00:00 to 2017-08-31 00:00:00


,id,date,store_nbr,item_nbr,onpromotion
0,125497040,2017-08-16,1,96995,0
1,125497041,2017-08-16,1,99197,0
2,125497042,2017-08-16,1,103501,0
3,125497043,2017-08-16,1,103520,0
4,125497044,2017-08-16,1,103665,0


In [19]:
# 1. Create the log-transformed target column in df_train
df_train['log_sales'] = np.log1p(df_train['unit_sales'])

# 2. Extract Day of Week (0 = Monday, 6 = Sunday) as a 1-byte integer
df_train['dayofweek'] = df_train['date'].dt.dayofweek.astype('int8')
df_test['dayofweek'] = df_test['date'].dt.dayofweek.astype('int8')

print(df_train[['date', 'dayofweek', 'unit_sales', 'log_sales']].head())

/tmp/ipykernel_22841/4037908186.py:5: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df_train['dayofweek'] = df_train['date'].dt.dayofweek.astype('int8')
/tmp/ipykernel_22841/4037908186.py:6: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df_test['dayofweek'] = df_test['date'].dt.dayofweek.astype('int8')


        date  dayofweek  unit_sales  log_sales
0 2016-01-01          4        12.0   2.564949
1 2016-01-01          4         9.0   2.302585
2 2016-01-01          4         3.0   1.386294
3 2016-01-01          4         3.0   1.386294
4 2016-01-01          4         2.0   1.098612


In [ ]:
from lightgbm import LGBMRegressor

# 1. Add basic calendar features to both train and test
for df in [df_train, df_test]:
    df['day'] = df['date'].dt.day.astype('int8')
    df['month'] = df['date'].dt.month.astype('int8')

features = ['store_nbr', 'item_nbr', 'onpromotion', 'dayofweek', 'day', 'month']

# 2. Initialize a fast Baseline LightGBM Regressor
# n_jobs=-1 uses all CPU cores on your ThinkPad
lgb_baseline = LGBMRegressor(
    n_estimators=100,
    learning_rate=0.1,
    num_leaves=31,
    random_state=42,
    n_jobs=6,             # Uses 6 threads for ML, leaves 2 threads free so your PC never lags!
    force_row_wise=True   # Removes the 0.8s multi-threading test overhead
)

# 3. Fit on the complete df_train (predicting log_sales optimizes NWRMSLE!)
print("Training LightGBM baseline on complete df_train...")
lgb_baseline.fit(
    df_train[features],
    df_train['log_sales'],
    categorical_feature=['store_nbr', 'item_nbr', 'dayofweek', 'month']
)

# 4. Predict on df_test and invert log-transform with np.expm1()
preds_log = lgb_baseline.predict(df_test[features])
df_test['unit_sales_lgb'] = np.expm1(preds_log).clip(lower=0)

print("Method B (LightGBM Baseline) sample predictions:")
display(df_test[['id', 'date', 'store_nbr', 'item_nbr', 'unit_sales_lgb']].head())

Training LightGBM baseline on complete df_train...
[LightGBM] [Warning] Met categorical feature which contains sparse values. Consider renumbering to consecutive integers started from zero
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.804997 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3767
[LightGBM] [Info] Number of data points in the train set: 59038132, number of used features: 6
[LightGBM] [Info] Start training from score 1.707525


TypeError: positive() got an unexpected keyword argument 'lower'

In [21]:
import joblib

# 1. Fix the clip syntax for NumPy arrays (use min=0 instead of lower=0)
df_test['unit_sales_lgb'] = np.expm1(preds_log).clip(min=0)

print("Method B (LightGBM Baseline) sample predictions:")
display(df_test[['id', 'date', 'store_nbr', 'item_nbr', 'unit_sales_lgb']].head(10))
print("Any NaNs left?", df_test['unit_sales_lgb'].isna().sum())
print(f"Min prediction: {df_test['unit_sales_lgb'].min():.4f} | Max prediction: {df_test['unit_sales_lgb'].max():.2f}")

# 2. Save the Kaggle submission CSV
df_test[['id', 'unit_sales_lgb']].rename(
    columns={'unit_sales_lgb': 'unit_sales'}
).to_csv('data/submission_lgb_baseline.csv', index=False)

# 3. Save the trained model to disk so you never lose those 9 minutes of work!
joblib.dump(lgb_baseline, 'data/lgb_baseline_model.pkl')
print("Saved submission CSV and model successfully!")

Method B (LightGBM Baseline) sample predictions:


,id,date,store_nbr,item_nbr,unit_sales_lgb
0,125497040,2017-08-16,1,96995,2.462142
1,125497041,2017-08-16,1,99197,2.462142
2,125497042,2017-08-16,1,103501,3.172452
3,125497043,2017-08-16,1,103520,1.923055
4,125497044,2017-08-16,1,103665,2.429007
5,125497045,2017-08-16,1,105574,4.807161
6,125497046,2017-08-16,1,105575,7.327540
7,125497047,2017-08-16,1,105576,8.677364
8,125497048,2017-08-16,1,105577,2.222511
9,125497049,2017-08-16,1,105693,1.677651


Any NaNs left? 0
Min prediction: 0.3645 | Max prediction: 489.99
Saved submission CSV and model successfully!


In [ ]:
# Save Method A submission
#sub_a[['id', 'unit_sales']].to_csv('data/submission_groupby_baseline.csv', index=False)

# Or save Method B (LightGBM) submission
df_test[['id', 'unit_sales_lgb']].rename(
    columns={'unit_sales_lgb': 'unit_sales'}
).to_csv('data/submission_lgb_baseline.csv', index=False)

print("Saved baseline submission files successfully!")

In [22]:
# 1. Load items.csv (4,100 rows)
df_items = pd.read_csv('data/items.csv')
df_items['item_nbr'] = df_items['item_nbr'].astype('int32')
df_items['class'] = df_items['class'].astype('int16')
df_items['perishable'] = df_items['perishable'].astype('int8')
# Convert text column 'family' into integer codes (0, 1, 2...32) for LightGBM
df_items['family'] = df_items['family'].astype('category').cat.codes.astype('int8')

# 2. Load stores.csv (54 rows)
df_stores = pd.read_csv('data/stores.csv')
df_stores['store_nbr'] = df_stores['store_nbr'].astype('int8')
df_stores['cluster'] = df_stores['cluster'].astype('int8')
# Keep raw city/state names temporarily for holiday matching, and create integer codes for ML
for col in ['city', 'state', 'type']:
    df_stores[col + '_code'] = df_stores[col].astype('category').cat.codes.astype('int8')

# 3. Load and clean oil.csv (fill missing weekend/holiday oil prices)
df_oil = pd.read_csv('data/oil.csv', parse_dates=['date']).sort_values('date')
df_oil['dcoilwtico'] = df_oil['dcoilwtico'].ffill().bfill().astype('float32')

# 4. Load and clean holidays_events.csv (remove transferred == True fake holidays)
df_holidays = pd.read_csv('data/holidays_events.csv', parse_dates=['date'])
df_active_holidays = df_holidays[df_holidays['transferred'] == False].copy()

print(f"Items shape: {df_items.shape} | Perishable items: {(df_items['perishable'] == 1).sum()}")
print(f"Stores shape: {df_stores.shape}")
print(f"Oil shape: {df_oil.shape} | Remaining oil NaNs: {df_oil['dcoilwtico'].isna().sum()}")
print(f"Active holidays rows: {len(df_active_holidays)}")

Items shape: (4100, 4) | Perishable items: 986
Stores shape: (54, 8)
Oil shape: (1218, 2) | Remaining oil NaNs: 0
Active holidays rows: 338


In [23]:
# 1. National holidays (apply to all dates)
nat = df_active_holidays[df_active_holidays['locale'] == 'National'][['date']].drop_duplicates()
nat['is_nat_holiday'] = np.int8(1)

# 2. Regional holidays (match locale_name to store state)
reg = df_active_holidays[df_active_holidays['locale'] == 'Regional'][['date', 'locale_name']].drop_duplicates()
reg = reg.merge(df_stores[['store_nbr', 'state']], left_on='locale_name', right_on='state')
reg = reg[['date', 'store_nbr']].drop_duplicates()
reg['is_reg_holiday'] = np.int8(1)

# 3. Local holidays (match locale_name to store city)
loc = df_active_holidays[df_active_holidays['locale'] == 'Local'][['date', 'locale_name']].drop_duplicates()
loc = loc.merge(df_stores[['store_nbr', 'city']], left_on='locale_name', right_on='city')
loc = loc[['date', 'store_nbr']].drop_duplicates()
loc['is_loc_holiday'] = np.int8(1)